# TraceAnalyzer — Demo & Fine-tuning Notebook

This notebook walks through the `TraceAnalyzer` component step by step:

1. **Generate traces** — play Sokoban games with `logging=True` to produce trace JSON files
2. **Inspect a trace** — understand the JSON structure (`metadata`, `moves`, `outcome`)
3. **Single-trace analysis** — call `TraceAnalyzer.analyze_single()` and read the LLM output
4. **Batch analysis** — call `TraceAnalyzer.analyze()` on multiple traces in parallel
5. **Runner integration check** — confirm `TraceAnalyzer` fires inside `OptimizationRunner`
6. **Fine-tuning** — experiment with `max_moves_per_trace`, custom `records_dir`, and disabling the analyzer via `use_trace_analyzer=False`

## Cell 1: Setup

In [15]:
import sys, json
from pathlib import Path

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mcts import MCTSEngine
from mcts.games import Sokoban
from LLM import TraceAnalyzer
from LLM.prompt_builder import PromptBuilder
from orchestrator import OptimizationRunner

RECORDS_DIR = ROOT / "mcts" / "records"
RECORDS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root          : {ROOT}")
print(f"Records dir   : {RECORDS_DIR}")
print("All imports OK ✓")

Root          : /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation
Records dir   : /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records
All imports OK ✓


## Cell 2: Generate Sokoban traces

Play a few games with `logging=True` so `MCTSEngine` writes trace JSON files.

In [16]:
LEVEL = "level4"
ITERS = 100
N_GAMES = 2

trace_files: list[Path] = []
for i in range(N_GAMES):
    game = Sokoban(LEVEL)
    engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=200, logging=True)
    result = engine.play_game()
    log_file = result.get("log_file", "")
    tag = "SOLVED" if result.get("solved") else "UNSOLVED"
    print(f"Game {i+1}: {tag} in {result.get('steps', '?')} steps → {Path(log_file).name if log_file else '(no log)'}")
    if log_file:
        trace_files.append(Path(log_file))

print(f"\n{len(trace_files)} trace file(s) generated.")

Game 1: SOLVED in 12 steps → Sokoban (level4)_20260310_182109_896707.json
Game 2: SOLVED in 12 steps → Sokoban (level4)_20260310_182109_945169.json

2 trace file(s) generated.


## Cell 3: Inspect trace structure

Each trace JSON has three top-level keys: `metadata`, `moves`, `outcome`.

In [17]:
trace_path = trace_files[0]
with open(trace_path) as f:
    trace = json.load(f)

print(f"File    : {trace_path.name}")
print(f"Keys    : {list(trace.keys())}")

print(f"\nMetadata:\n{json.dumps(trace.get('metadata', {}), indent=2)}")
print(f"\nOutcome :\n{json.dumps(trace.get('outcome', {}), indent=2)}")

moves = trace.get("moves", [])
print(f"\nMoves   : {len(moves)} total")
if moves:
    print(f"Move keys  : {list(moves[0].keys())}")
    print(f"\nFirst move:\n{json.dumps(moves[0], indent=2)[:800]}")

File    : Sokoban (level4)_20260310_182109_896707.json
Keys    : ['metadata', 'moves', 'outcome']

Metadata:
{
  "game": "Sokoban (level4)",
  "timestamp": "2026-03-10T18:21:09.848292",
  "iterations": 100,
  "max_rollout_depth": 200,
  "exploration_weight": 1.41,
  "tools": {
    "selection": "/Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/selection/default_selection.py",
    "expansion": "/Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/expansion/default_expansion.py",
    "simulation": "/Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/simulation/default_simulation.py",
    "backpropagation": "/Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/backpropagation/default_backpropagation.py"
  }
}

Outcome :
{
  "solved": true,
  "steps": 12,
  "returns": [
    1.0
  ],
  "final_state": "Step 12/200 | Boxes on target: 2

## Cell 4: Single-trace analysis

`analyze_single()` sends one trace to the LLM and returns a natural-language
description covering outcome, key moments, root cause, and heuristic insight.

Make sure Ollama is running (`ollama serve`).

In [18]:
analyzer = TraceAnalyzer(game="sokoban")

print(f"Analyzing: {trace_files[0].name} …")
single_analysis = analyzer.analyze_single(trace_files[0])

print("\n=== Single-trace LLM analysis ===")
print(single_analysis)

Analyzing: Sokoban (level4)_20260310_182109_896707.json …

=== Single-trace LLM analysis ===
**1. Outcome Summary**  
- **Result:** Win – the agent solved the level.  
- **Moves taken:** 12 actions (steps 0‑11). The final return is 1.0, indicating all boxes were on targets.

**2. Key Moments**  

| Move | What happened & why it mattered |
|------|---------------------------------|
| **1‑4 (actions 3, 3, 0, 1)** | The agent moved right twice, then up and down to thread itself behind the right‑hand box. This positioning created a clear line of push‑space that would later let the box be driven onto its target. Without getting into that corridor, the box would have been stuck against the wall. |
| **9 (action 0)** | The player stepped onto the square directly left of the right‑most box and pushed it up onto the target (the `*`). Boxes‑on‑targets jumped from 0/2 to 1/2 and the total distance fell from 2 to 1. This was the decisive “first‑goal” achievement that turned the search from explora

## Cell 5: Batch analysis (parallel)

`analyze()` runs all traces in parallel via `query_batch` and concatenates the results.
This is the string that gets injected as `additional_context` in the runner.

In [19]:
print(f"Analyzing {len(trace_files)} traces in parallel…")
batch_analysis = analyzer.analyze(trace_files)

print("\n=== Batch analysis ===")
print(batch_analysis[:1500], "…" if len(batch_analysis) > 1500 else "")
print(f"\nTotal length: {len(batch_analysis)} chars")

Analyzing 2 traces in parallel…

=== Batch analysis ===
=== Trace #1 Analysis ===
**1. Outcome Summary**  
- **Result:** Win – the puzzle was solved (return = 1.0).  
- **Moves taken:** 12 player actions (Step 0 → Step 11).

**2. Key Moments**  

| Move | What happened | Why it mattered |
|------|----------------|-----------------|
| **1‑2** (actions 3, 3) | The agent moved right twice, positioning the player next to the left‑hand box and then stepping onto the target (the “+” symbol). This set up a clear line of sight to the first box without disturbing the other box. | Establishes a safe corridor and avoids premature pushes; the early “+” shows the player is already on a target, keeping the reward gradient high. |
| **9** (action 0) | The player moved up and pushed the leftmost box onto its target (the “*”). Boxes on target rose from 0/2 to 1/2 and total distance dropped from 2 to 1. | The only decisive push of the game; it converts distance‑based reward into a concrete target and cr

## Cell 6: OptimizationRunner integration check

Run **one iteration** of the optimizer to confirm `TraceAnalyzer` fires automatically
inside the loop.  Look for the `TraceAnalyzer:` lines in the verbose output.

In [20]:
SOKOBAN_CONFIG = {
    "game_name": "sokoban",
    "game_class": "Sokoban",
    "game_module": "mcts.games",
    "training_logic": "sokoban_training",
    "phases": ["simulation"],
    "num_iters": 1,              # single iteration — quick smoke test
    "three_step": False,         # two-step is faster here
    "logging": True,
    "use_trace_analyzer": True,  # ← THE FEATURE UNDER TEST
    "hyperparams": {
        "iterations": 50,
        "max_rollout_depth": 100,
        "exploration_weight": 1.41,
    },
}

runner = OptimizationRunner.from_dict(SOKOBAN_CONFIG, verbose=True)

print(f"runner._trace_analyzer : {type(runner._trace_analyzer).__name__}")
print(f"runner._trace_analyzer.game: {runner._trace_analyzer.game}")
print(f"runner.use_trace_analyzer  : {runner.use_trace_analyzer}")
print()

summary = runner.run()

print("\nIteration records:")
for r in summary["all_results"]:
    print(f"  iter={r['iteration']}  level={r['level']}  "
          f"smoke={r['smoke_test']}  composite={r['composite']:.4f}")

  Resuming simulation from previously installed tool.
runner._trace_analyzer : TraceAnalyzer
runner._trace_analyzer.game: sokoban
runner.use_trace_analyzer  : True

Starting level: level4
Hyperparams: iterations=50, max_depth=100, C=1.410
Phases to optimize: ['simulation']
  Computing baseline for level4…
    level4: composite=0.6667, solve_rate=67%, avg_returns=0.6667 (0.5s)
  Reject floor for level4: 0.3333
  Active levels: ['level4', 'level5', 'level6', 'level7', 'level8']

############################################################
  ITERATION 1/1, LEVEL=level4, PHASE=simulation
  Baseline composite=0.6667, reject_floor=0.3333
  Active levels: 5/5, mastered: []
############################################################
  Play: SOLVED in 8 steps  returns=1.0000  (0.4s)
  Analyzing trace with TraceAnalyzer…
  Trace analysis: 1766 chars
Step 1/4: Querying LLM (step 1 — analysis)…
Step 2/4: Querying LLM (step 2 — code generation)…
  LLM query: status=success  elapsed=45.95s
  Step-1

## Cell 7: Fine-tuning options

Experiment with `TraceAnalyzer` parameters:

| Parameter | Effect |
|-----------|--------|
| `max_moves_per_trace=N` | Truncates long traces to N moves in the prompt |
| `records_dir=path` | Override where traces are loaded from |
| `use_trace_analyzer=False` | Disable entirely in the runner (saves one LLM call per iteration) |

The cell below compares prompt sizes for different `max_moves_per_trace` values.

In [21]:
from LLM.trace_analyzer import TraceAnalyzer as TA

print("max_moves_per_trace  |  prompt length (chars)")
print("-" * 44)
for cap in [None, 40, 20, 10]:
    ta = TA(game="sokoban", max_moves_per_trace=cap)
    with open(trace_files[0]) as f:
        trace = json.load(f)
    game_rules = ta._load_game_info()
    prompt = ta._build_single_trace_prompt(game_rules, trace, index=1)
    label = str(cap) if cap is not None else "None (all)"
    print(f"{label:>20}  |  {len(prompt):,}")

print()
print("Disable in runner → pass  use_trace_analyzer=False  to from_dict().")

max_moves_per_trace  |  prompt length (chars)
--------------------------------------------
          None (all)  |  7,247
                  40  |  7,247
                  20  |  7,247
                  10  |  6,826

Disable in runner → pass  use_trace_analyzer=False  to from_dict().


# Trace Analyzer & Prompt Builder — Step-by-Step Test (Sokoban)

This notebook exercises the record analyzer pipeline one piece at a time:

1. **Cell 1** — imports & setup
2. **Cell 2** — generate Sokoban trace files via `MCTSEngine.play_game(logging=True)`
3. **Cell 3** — inspect a raw trace JSON
4. **Cell 4** — `PromptBuilder` with `verbose=True` (see every section logged)
5. **Cell 5** — `TraceAnalyzer.analyze_single()` on one trace
6. **Cell 6** — `TraceAnalyzer.analyze()` batch on multiple traces

## Cell 1: Setup

In [8]:
import sys, json
from pathlib import Path

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mcts import MCTSEngine
from mcts.games import Sokoban
from LLM import PromptBuilder, TraceAnalyzer, LLMQuerier

RECORDS_DIR = ROOT / "mcts" / "records"
RECORDS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root        : {ROOT}")
print(f"Records dir : {RECORDS_DIR}")
print("All imports OK ✓")

Root        : /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation
Records dir : /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records
All imports OK ✓


## Cell 2: Generate Sokoban trace files

Play 2 games with logging enabled to produce trace JSONs that the analyzer needs.

In [9]:
LEVEL = "level4"
ITERS = 100

trace_files = []
for i in range(2):
    game = Sokoban(LEVEL)
    engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=200, logging=True)
    result = engine.play_game()
    log_file = result.get("log_file")
    tag = "SOLVED" if result.get("solved") else "UNSOLVED"
    print(f"Game {i+1}: {tag} in {result.get('steps','?')} steps → {log_file}")
    if log_file:
        trace_files.append(Path(log_file))

print(f"\n{len(trace_files)} trace file(s) generated.")

Game 1: SOLVED in 12 steps → /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records/Sokoban (level4)_20260310_175554_835741.json
Game 2: SOLVED in 10 steps → /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records/Sokoban (level4)_20260310_175554_871090.json

2 trace file(s) generated.


## Cell 3: Inspect a raw trace JSON

Look at the structure of one trace to understand what the analyzer sees.

In [10]:
trace_path = trace_files[0]
with open(trace_path) as f:
    trace = json.load(f)

print(f"File  : {trace_path.name}")
print(f"Keys  : {list(trace.keys())}")
meta = trace.get("metadata", {})
outcome = trace.get("outcome", {})
print(f"\nMetadata  : {json.dumps(meta, indent=2)}")
print(f"\nOutcome   : {json.dumps(outcome, indent=2)}")
moves = trace.get("moves", [])
print(f"\nMoves : {len(moves)} total")
if moves:
    print(f"\nFirst move keys: {list(moves[0].keys())}")
    print(f"First move sample:\n{json.dumps(moves[0], indent=2)[:600]}")


File  : Sokoban (level4)_20260310_175554_835741.json
Keys  : ['metadata', 'moves', 'outcome']

Metadata  : {
  "game": "Sokoban (level4)",
  "timestamp": "2026-03-10T17:55:54.778585",
  "iterations": 100,
  "max_rollout_depth": 200,
  "exploration_weight": 1.41,
  "tools": {
    "selection": "/Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/selection/default_selection.py",
    "expansion": "/Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/expansion/default_expansion.py",
    "simulation": "/Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/simulation/default_simulation.py",
    "backpropagation": "/Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/MCTS_tools/backpropagation/default_backpropagation.py"
  }
}

Outcome   : {
  "solved": true,
  "steps": 12,
  "returns": [
    1.0
  ],
  "final_state": "Step 12/200 | Boxes on target: 2

## Cell 4: PromptBuilder with `verbose=True`

Build both the **analysis prompt** (step 1) and the **full prompt** (legacy) while
watching the verbose log — each section's name and character count is printed.

In [11]:
builder = PromptBuilder(
    game="sokoban",
    target_phase="simulation",
    verbose=True,           # ← logs each section as it's assembled
)

print("=== build_analysis_prompt() ===")
analysis_prompt = builder.build_analysis_prompt(
    record_files=trace_files,
    max_moves_per_trace=20,
)

print()
print("=== build() (legacy single-step prompt) ===")
full_prompt = builder.build(
    record_files=trace_files,
    max_moves_per_trace=20,
)

print()
print("--- Analysis prompt preview (first 500 chars) ---")
print(analysis_prompt[:500])

=== build_analysis_prompt() ===
  [system]  1,131 chars, 26 lines
  [game_rules]  3,546 chars, 85 lines
  [tool_source]  213 chars, 4 lines
  [traces(2)]  4,495 chars, 103 lines
  [analysis_task]  1,188 chars, 33 lines
  [total analysis_prompt]  10,581 chars

=== build() (legacy single-step prompt) ===
  [system]  1,131 chars, 26 lines
  [game_rules]  3,546 chars, 85 lines
  [tool_source]  213 chars, 4 lines
  [traces(2)]  4,495 chars, 103 lines
  [task]  1,974 chars, 50 lines
  [total prompt]  11,367 chars

--- Analysis prompt preview (first 500 chars) ---
SYSTEM: MCTS Heuristic Improvement
You are an expert game-playing AI researcher.
Your task is to improve a specific MCTS heuristic function
for the game 'sokoban' (phase: simulation).

APPROACH — 70 / 30 RULE:
  ~70% of iterations: INCREMENTAL OPTIMIZATION
    • Start from the CURRENT code.
    • Make targeted, gradual improvements (add a check,
      tweak weights, add a heu


## Cell 5: TraceAnalyzer — single trace

Ask the LLM to explain what happened in one game trace.
Make sure Ollama is running (`ollama serve`).

In [12]:
analyzer = TraceAnalyzer(game="sokoban")

print(f"Analyzing trace: {trace_files[0].name}")
single_analysis = analyzer.analyze_single(trace_files[0])

print("\n=== Single-trace LLM analysis ===")
print(single_analysis)

Analyzing trace: Sokoban (level4)_20260310_175554_835741.json

=== Single-trace LLM analysis ===
**1. Outcome Summary**  
- **Result:** Victory – the level was solved (return = 1.0).  
- **Length:** 12 actions (moves 0‑11) out of a 200‑step limit.

**2. Key Moments**  

| Move | What happened & why it mattered |
|------|---------------------------------|
| **7 (action = 3)** – push right | The player moves next to the left‑hand box and pushes it onto the nearest target, raising *boxes‑on‑target* from 0 to 1 and dropping the total distance from 2 to 1. This creates a clear “half‑solved” state and gives the planner a high‑value subtree (q≈0.11 for both push directions). |
| **10 (action = 2)** – walk left | After the first box is secured, the agent redirects toward the second box. The visit count for action 2 spikes to 80 with a Q‑value of 0.66, indicating the search strongly favored moving left to line up a push. This decisive repositioning sets up the final push. |
| **11 (action = 2)*

## Cell 6: TraceAnalyzer — batch (parallel)

Analyze both traces in parallel and get a combined context string
ready to pass as `additional_context` to the optimizer.

In [13]:
print(f"Analyzing {len(trace_files)} traces in parallel…")
batch_analysis = analyzer.analyze(trace_files)

print("\n=== Batch LLM analysis ===")
print(batch_analysis)

print(f"\n--- Summary ---")
print(f"Total chars   : {len(batch_analysis):,}")
print(f"Suitable for  : builder.build_analysis_prompt(..., additional_context=batch_analysis)")

Analyzing 2 traces in parallel…

=== Batch LLM analysis ===
=== Trace #1 Analysis ===
**1. OUTCOME SUMMARY**  
- **Result:** Win – the puzzle was solved (return = 1.0).  
- **Moves taken:** 12 actions (steps 0‑11).  

**2. KEY MOMENTS**  

| Move | What happened & why it mattered |
|------|---------------------------------|
| **7** | The agent pushed the right‑hand box onto its target (boxes‑on‑target = 1/2, distance ↓). This was the first “hard” progress and set a safe anchor for later moves. |
| **10** | After circling around the lower corridor, the player positioned directly behind the remaining box (see the “@” on the left side). This created a clear line‑of‑push toward the second target and avoided getting trapped against the wall. |
| **12** | A final downward push (action = 1) moved the last box onto its goal, completing the level. The move also kept the player away from any corner dead‑lock, confirming the earlier positioning was correct. |

**3. WHY IT WORKED**  
The agent fol

## Cell 7: TraceAnalyzer embedded in OptimizationRunner

Verify that `use_trace_analyzer=True` (the default) wires a `TraceAnalyzer`
into the runner, and watch it fire during a single iteration.


In [14]:
from orchestrator.runner import OptimizationRunner

SOKOBAN_CONFIG = {
    "game_name": "sokoban",
    "game_class": "Sokoban",
    "training_logic": "sokoban_training",
    "phases": ["simulation"],
    "num_iters": 1,          # single iteration so we can inspect the output
    "three_step": False,     # two-step is faster for a quick test
    "logging": True,
    "use_trace_analyzer": True,   # ← this is now the default
    "hyperparams": {
        "iterations": 50,
        "max_rollout_depth": 100,
        "exploration_weight": 1.41,
    },
}

runner = OptimizationRunner.from_dict(SOKOBAN_CONFIG, verbose=True)

# Check the analyzer was created
print(f"_trace_analyzer type : {type(runner._trace_analyzer)}")
print(f"_trace_analyzer game : {runner._trace_analyzer.game}")
print()

# Run 1 iteration — look for "Analyzing trace with TraceAnalyzer…" in the output
summary = runner.run()
print("\nIteration records:")
for r in summary["all_results"]:
    print(f"  iter={r['iteration']} solved={r['smoke_test']} "
          f"composite={r['composite']:.4f}")


  Resuming simulation from previously installed tool.
_trace_analyzer type : <class 'LLM.trace_analyzer.TraceAnalyzer'>
_trace_analyzer game : sokoban

Starting level: level4
Hyperparams: iterations=50, max_depth=100, C=1.410
Phases to optimize: ['simulation']
  Computing baseline for level4…
    level4: composite=0.6667, solve_rate=67%, avg_returns=0.6667 (0.3s)
  Reject floor for level4: 0.3333
  Active levels: ['level4', 'level5', 'level6', 'level7', 'level8']

############################################################
  ITERATION 1/1, LEVEL=level4, PHASE=simulation
  Baseline composite=0.6667, reject_floor=0.3333
  Active levels: 5/5, mastered: []
############################################################
  Play: SOLVED in 18 steps  returns=1.0000  (2.8s)
  Analyzing trace with TraceAnalyzer…
  Trace analysis: 1948 chars
Step 1/4: Querying LLM (step 1 — analysis)…
Step 2/4: Querying LLM (step 2 — code generation)…
  LLM query: status=success  elapsed=36.83s
  Step-1 analysis le

In [7]:
# Quick sanity check on the runner built above
print(f"runner._trace_analyzer : {runner._trace_analyzer}")
print(f"runner._trace_analyzer type : {type(runner._trace_analyzer).__name__}")
print(f"runner._trace_analyzer.game : {runner._trace_analyzer.game}")
print(f"runner.use_trace_analyzer  : {runner.use_trace_analyzer}")
print()
print("TraceAnalyzer is embedded in OptimizationRunner ✓")


runner._trace_analyzer : <LLM.trace_analyzer.TraceAnalyzer object at 0x107ac2870>
runner._trace_analyzer type : TraceAnalyzer
runner._trace_analyzer.game : sokoban
runner.use_trace_analyzer  : True

TraceAnalyzer is embedded in OptimizationRunner ✓
